In [26]:
import urllib.request
import pandas as pd
import json

base_url = "https://www.donneesquebec.ca/recherche/api/3/action/datastore_search"
resource_id = "c6640a54-bc4b-43ec-864e-6c325dce61bc"
filters = json.dumps({"NOM_MUNCP_GDUNO_ORGNS": "Montréal"})
# filters = {}
query_params = urllib.parse.urlencode(
    {"resource_id": resource_id, "filters": filters, "limit": 5000}
)
url = f"{base_url}?{query_params}"
with urllib.request.urlopen(url) as response:
    data_dict = json.loads(response.read().decode('utf-8'))
df = pd.DataFrame(data_dict["result"]["records"])

df

,_id,DT_MAJ_GDUNO,COMBINE_NUO_NUI,CD_ORGNS,NOM_COURT_ORGNS,NOM_OFFCL_ORGNS,ADRS_GEO_L1_GDUNO_ORGNS,ADRS_GEO_L2_GDUNO_ORGNS,CD_POSTL_GDUNO_ORGNS,CD_MUNCP_GDUNO_ORGNS,...,COORD_Y_LL84_IMM,ORDRE_ENS,CD_CS,TYPE_CS,STYLE_CART,NOM_MUNCP,NOM_MRC,NOM_REG_ADM,NOM_CEP,NOM_CS
0,48,None,None,762132,École François-de-Laval,École François-de-Laval,"12050, avenue Bois-de-Boulogne",NaN,H3M2X9,66023,...,45.542872,Préscolaire - Primaire,762000,Franco,4_OrgImm_Franco_presc_prim,Montréal,Montréal,Montréal,Acadie,Centre de services scolaire de Montréal
1,156,None,None,763405,CFP Léonard-De Vinci,Centre de formation professionnelle Léonard-De...,"2405, boulevard Thimens",NaN,H4R1T4,66023,...,45.50977,Formation professionnelle,763000,Franco,8_OrgImm_Franco_Fp,Montréal,Montréal,Montréal,Nelligan,Centre de services scolaire Marguerite-Bourgeoys
2,157,None,None,763245,École de l'Altitude,École de l'Altitude,"5060, boulevard des Sources",NaN,H8Y3E4,66023,...,45.50977,Secondaire,763000,Franco,3_OrgImm_Franco_secondaire,Montréal,Montréal,Montréal,Nelligan,Centre de services scolaire Marguerite-Bourgeoys
3,237,None,None,887253,École Centre Mackay,École Centre Mackay,"6333, rue de Terrebonne",NaN,H4B1A8,66023,...,45.468946,Préscolaire - Primaire - Secondaire,887000,Anglo,5_OrgImm_Anglo_presc_prim_sec,Montréal,Montréal,Montréal,Notre-Dame-de-Grâce,Commission scolaire English-Montréal
4,238,None,None,887256,École Philip E. Layton,École Philip E. Layton,"6333, rue de Terrebonne",NaN,H4B1A8,66023,...,45.468946,Préscolaire - Primaire - Secondaire,887000,Anglo,5_OrgImm_Anglo_presc_prim_sec,Montréal,Montréal,Montréal,Notre-Dame-de-Grâce,Commission scolaire English-Montréal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566,5403,None,None,888310,Centre de formation des adultes West-Isl,Centre de formation des adultes West-Island,"13700, boulevard de Pierrefonds",NaN,H9A1A7,66023,...,45.49249,Éducation aux adultes,888000,Anglo,9_OrgImm_Anglo_Adu,Montréal,Montréal,Montréal,Robert-Baldwin,Commission scolaire Lester-B.-Pearson
567,5415,None,None,888401,Centre PACC-Formation professionnelle,Centre de formation des adultes et de formatio...,"8310, rue George",NaN,H8P1E5,66023,...,45.451981,Formation professionnelle,888000,Anglo,8_OrgImm_Anglo_Fp,Pointe-Claire,Montréal,Montréal,Jacques-Cartier,Commission scolaire Lester-B.-Pearson
568,5416,None,None,888401,Centre PACC-Formation professionnelle,Centre de formation des adultes et de formatio...,"8310, rue George",NaN,H8P1E5,66023,...,45.4222,Formation professionnelle,888000,Anglo,8_OrgImm_Anglo_Fp,Montréal,Montréal,Montréal,Marguerite-Bourgeoys,Commission scolaire Lester-B.-Pearson
569,5417,None,None,888403,Centre de formation prof. West-Island,Centre de formation professionnelle West-Island,"13700, boulevard de Pierrefonds",NaN,H9A1A7,66023,...,45.49249,Formation professionnelle,888000,Anglo,8_OrgImm_Anglo_Fp,Montréal,Montréal,Montréal,Robert-Baldwin,Commission scolaire Lester-B.-Pearson


In [27]:
from borough_identifier import BoroughIdentifier

bi = BoroughIdentifier()
x = df['COORD_X_LL84_IMM']
y = df['COORD_Y_LL84_IMM']
# X and Y are switched
df['borough'] = df.apply(lambda row: bi.match_WSG84_to_borough(longitude_x=row['COORD_X_LL84_IMM'], latitude_y=row['COORD_Y_LL84_IMM']), axis=1)

In [28]:
df = df.groupby(['NOM_OFFCL_ORGNS', 'borough'], as_index=False).agg(lambda x: ', '.join(x.dropna().astype(str).unique()))

In [29]:
df['ORDRE_ENS'].unique()

<StringArray>
[                                                 'Secondaire',
                                      'Préscolaire - Primaire',
                          'Préscolaire - Primaire, Secondaire',
                                       'Éducation aux adultes',
                                   'Formation professionnelle',
                         'Préscolaire - Primaire - Secondaire',
            'Éducation aux adultes, Formation professionnelle',
                            'Préscolaire - Primaire, Primaire',
                                                    'Primaire',
                                                 'Préscolaire',
             'Préscolaire - Primaire - Secondaire, Secondaire',
                                       'Primaire - Secondaire',
 'Préscolaire - Primaire, Préscolaire - Primaire - Secondaire',
                            'Primaire, Préscolaire - Primaire',
  'Primaire - Secondaire, Préscolaire - Primaire - Secondaire',
                         '

In [30]:
df = df[~df['ORDRE_ENS'].str.contains('adultes')]
df = df[~df['ORDRE_ENS'].str.contains('professionnelle')]

In [31]:
df['borough'].unique()

<StringArray>
[    'Villeray-Saint-Michel-Parc-Extension',
            'Mercier-Hochelaga-Maisonneuve',
      'Côte-des-Neiges-Notre-Dame-de-Grâce',
                    'Ahuntsic-Cartierville',
 'Rivière-des-Prairies-Pointe-aux-Trembles',
                            'Montréal-Nord',
                'Rosemont-La Petite-Patrie',
                    'Le Plateau-Mont-Royal',
                                  'Lachine',
                              'Ville-Marie',
                                  'LaSalle',
                                    'Anjou',
                            'Saint-Léonard',
                            'Saint-Laurent',
                                   'Verdun',
                             'Le Sud-Ouest',
                                'Outremont',
                      'Pierrefonds-Roxboro',
            'L'Île-Bizard-Sainte-Geneviève']
Length: 19, dtype: str

In [32]:
for brgh in df['borough'].unique():
    schools = len(df[df['borough']==brgh])
    print(f"{brgh}: {schools} schools")

Villeray-Saint-Michel-Parc-Extension: 34 schools
Mercier-Hochelaga-Maisonneuve: 31 schools
Côte-des-Neiges-Notre-Dame-de-Grâce: 36 schools
Ahuntsic-Cartierville: 27 schools
Rivière-des-Prairies-Pointe-aux-Trembles: 24 schools
Montréal-Nord: 17 schools
Rosemont-La Petite-Patrie: 32 schools
Le Plateau-Mont-Royal: 20 schools
Lachine: 13 schools
Ville-Marie: 9 schools
LaSalle: 21 schools
Anjou: 8 schools
Saint-Léonard: 11 schools
Saint-Laurent: 15 schools
Verdun: 15 schools
Le Sud-Ouest: 21 schools
Outremont: 6 schools
Pierrefonds-Roxboro: 12 schools
L'Île-Bizard-Sainte-Geneviève: 3 schools


In [33]:
total_pop = {
    'Côte-des-Neiges-Notre-Dame-de-Grâce': 170583, # according to https://ville.montreal.qc.ca/pls/portal/docs/PAGE/MTL_STATS_FR/MEDIA/DOCUMENTS/CARTE_POPULATION%20ET%20SUPERFICIE%202021.PDF
    'Ville-Marie': 104944,
    'Montréal-Nord': 88471,
    'Mercier-Hochelaga-Maisonneuve': 140627,
    'Pierrefonds-Roxboro': 70382,
    'Rivière-des-Prairies-Pointe-aux-Trembles': 107941,
    'Lachine': 46428,
    'Saint-Laurent': 102104,
    'Rosemont-La Petite-Patrie': 141813,
    'Ahuntsic-Cartierville': 135336,
    'Le Plateau-Mont-Royal': 105813,
    'Verdun': 70377,
    'LaSalle': 82235,
    'Villeray-Saint-Michel-Parc-Extension': 145090,
    'Anjou': 43243,
    'Le Sud-Ouest': 84553,
    'L\'Île-Bizard-Sainte-Geneviève': 18885,
    'Saint-Léonard': 79495,
    'Outremont': 24629
}

for brgh in df['borough'].unique():
    schools = len(df[df['borough']==brgh])
    print(f"{brgh}: {((total_pop[brgh] / schools)):.2f} people per school.")

Villeray-Saint-Michel-Parc-Extension: 4267.35 people per school.
Mercier-Hochelaga-Maisonneuve: 4536.35 people per school.
Côte-des-Neiges-Notre-Dame-de-Grâce: 4738.42 people per school.
Ahuntsic-Cartierville: 5012.44 people per school.
Rivière-des-Prairies-Pointe-aux-Trembles: 4497.54 people per school.
Montréal-Nord: 5204.18 people per school.
Rosemont-La Petite-Patrie: 4431.66 people per school.
Le Plateau-Mont-Royal: 5290.65 people per school.
Lachine: 3571.38 people per school.
Ville-Marie: 11660.44 people per school.
LaSalle: 3915.95 people per school.
Anjou: 5405.38 people per school.
Saint-Léonard: 7226.82 people per school.
Saint-Laurent: 6806.93 people per school.
Verdun: 4691.80 people per school.
Le Sud-Ouest: 4026.33 people per school.
Outremont: 4104.83 people per school.
Pierrefonds-Roxboro: 5865.17 people per school.
L'Île-Bizard-Sainte-Geneviève: 6295.00 people per school.


In [34]:
output = {
    'borough': df['borough'].unique(),
}
output_df = pd.DataFrame(output)
output_df['total_pop'] = output_df['borough'].map(total_pop)
output_df['num_schools'] = output_df['borough'].map(lambda brgh: len(df[df['borough']==brgh]))
output_df['residents_per_school'] = output_df['total_pop'] / output_df['num_schools']
output_df

,borough,total_pop,num_schools,residents_per_school
0,Villeray-Saint-Michel-Parc-Extension,145090,34,4267.352941
1,Mercier-Hochelaga-Maisonneuve,140627,31,4536.354839
2,Côte-des-Neiges-Notre-Dame-de-Grâce,170583,36,4738.416667
3,Ahuntsic-Cartierville,135336,27,5012.444444
4,Rivière-des-Prairies-Pointe-aux-Trembles,107941,24,4497.541667
5,Montréal-Nord,88471,17,5204.176471
6,Rosemont-La Petite-Patrie,141813,32,4431.656250
7,Le Plateau-Mont-Royal,105813,20,5290.650000
8,Lachine,46428,13,3571.384615
9,Ville-Marie,104944,9,11660.444444


In [35]:
output_df.to_csv('../data/schools_dataset.csv', index=False)